Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.


# Overall of this notebook

Most of concepts and codes are adapted from
- https://github.com/dair-ai/Prompt-Engineering-Guide
- https://ai.google.dev/gemini-api/docs/prompting-strategies
- https://myframework.net/icio-ai-prompt-framework/

# Setting environments and model setup

In [ ]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [1]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


In [2]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    reasoning_effort="none",
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Basic

## System Prompt / User Prompt

`System Prompt`:

The system prompt establishes the overall context, persona, and behavioral guidelines for the LLM. It dictates how the model should generally respond and interact, setting the foundational rules for all subsequent interactions within a session or application.

`User Prompt (Human)`:

  The user prompt is the specific query or instruction provided by the user to the LLM. It defines the immediate task or question the user wants the model to address, operating within the framework established by the system prompt. example


In [22]:
# Demo 1: System - Health Scientist / User - Explain the importance of exercise
messages = [
    ("system", "You are a health scientist who always provides factual and evidence-based answers."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Regular physical activity is essential for maintaining cardiovascular health, metabolic function, and mental well-being while significantly reducing the risk of chronic diseases.

In [23]:
# Demo 2: Syetem - Elderly Person Complaining / User - Explain the importance of exercise
messages = [
    ("system", "You are an elderly person who often complains."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Ugh, the doctors keep nagging me to move my bones before they turn to dust, but honestly, all I want is to sit in this chair and complain.

In [25]:
# Demo 3: System - Mother Explaining to a 5-year-old / User - Explain the importance of exercise
messages = [
    ("system", "You are a mother who needs to answer questions from a 5-year-old child, always explaining complex topics in the simplest way possible."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Exercise helps your body grow big and strong so you can run, jump, and play all day without getting tired!

In [26]:
# Demo 4: Professional Assistant (Python factorial)
messages = [
    ("system", "You are a helpful and informative assistant. Your responses should be clear, concise, and professional. Avoid making assumptions and always ask for clarification if a user's request is ambiguous."),
    ("human", "Write a Python function that calculates the factorial of a given number."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

```python
def factorial(n: int) -> int:
    """
    Calculate the factorial of a non-negative integer.
    
    Args:
        n: A non-negative integer.
        
    Returns:
        The factorial of n.
        
    Raises:
        ValueError: If n is negative.
        TypeError: If n is not an integer.
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer.")
    if n < 0:
        raise ValueError("Input must be a non-negative integer.")
    if n == 0 or n == 1:
        return 1
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

## User Prompt Framework - ICIO

The ICIO framework is a simple and practical method that **helps you structure your prompts** step by step.
- `Instruction (I)` --> What do you want the AI to do?

  - The instruction should be specific and direct. A clear task helps the AI give you the right kind of output.
- `Context (C)` --> Give background information. Why are you doing this task? What’s the situation?

  - Context helps the AI better understand your purpose and tone.
  - ***Optional, but nice to have.***

- `Input (I)` --> What exact text or data should the AI process?
  - Provide the content the AI needs to work with.
  - Without input data, the AI may guess or go off track. Be clear and complete.

- `Output (O)` --> Set the style or format of the output. What should the response look like? What tone or structure do you expect?
  - This helps guide the AI to produce the kind of result you want.

In [12]:
# Example A: Customer Feedback Summary

# ICIO fields
instruction = "Summarize customer opinions."
context = "For the product development team to consider improvements in the next version."
input_text = (
    "Many customers like the battery lasting up to 3 days, which is much better than the older version. "
    "The AMOLED screen provides vibrant colors and is clearly visible even under bright sunlight. "
    "However, some users reported that Bluetooth connections with certain headphones often drop. "
    "The sleep tracking system is also not very accurate and sometimes fails to record data. "
    "Additionally, customers would like to see a blood pressure monitoring function added."
)
output_format = 'Summarize into a table with 3 columns: "Feature", "Status (Good/Needs Improvement/Requested)", "Notes".'

# Create messages
messages = [
    ("system", "You are a product analyst who summarizes customer feedback into clear, structured tables for business teams."),
    ("human",
     f"{instruction}\n"
     f"{context}\n"
     f"{input_text}\n"
     f"{output_format}"
    ),
]

# Invoke model
ai_msg = llm.invoke(messages)

In [13]:
from IPython.display import display, Markdown
display(Markdown(ai_msg.content))

| Feature | Status (Good/Needs Improvement/Requested) | Notes |
| :--- | :--- | :--- |
| Battery Life | Good | Lasts up to 3 days; significant improvement over the older version. |
| Display (AMOLED) | Good | Provides vibrant colors and remains clearly visible under bright sunlight. |
| Bluetooth Connectivity | Needs Improvement | Connections with certain headphones often drop. |
| Sleep Tracking | Needs Improvement | Data accuracy is low; system sometimes fails to record data. |
| Blood Pressure Monitoring | Requested | Customers are requesting this function be added in the next version. |

***To summarize, recognizing ICIO when prompting helps ensure the prompt is complete and clear, and that the LLM provides the desired output.***

## Structured Input

In prompt engineering, **structured input** helps guide the LLM to focus on exactly what we want.  

One common technique is using **delimiters** (special symbols or markers) to clearly separate instructions, context, and input data.


Why use delimiters?
- They **reduce ambiguity** → the model doesn’t “guess” where instructions or content begin/end.  
- They **minimize misinterpretation** → the model treats the content inside delimiters as a defined block.  
- They are especially useful when prompts are **long, multi-part, or contain different types of information**.
---

Examples of delimiters

You can use different symbols such as:
- Triple dashes (---)
- Triple hashtags (###)
- Triple backticks: \`\`\` ... \`\`\`
- Triple quotes: """ ... """
- Angle brackets: < ... >
- Tags: `<instruction> ... </instruction>`

In [18]:
# The raw text to be summarized
text = """
In the digital age, online marketing has become the cornerstone of businesses of all sizes, offering a broad reach to consumers at a lower cost than traditional marketing.
Popular online marketing tools include SEO (Search Engine Optimization), Social Media Marketing, and high-quality Content Marketing.
Leveraging data analytics also helps businesses analyze customer behavior and refine their strategies effectively.
"""

# The prompt using delimiters (triple backticks ```)
prompt = f"""You are a helpful assistant.
Summarize the text within the triple backticks concisely, in no more than two sentences.

```{text}```
"""

ai_msg = llm.invoke(prompt)

In [19]:
display(Markdown(ai_msg.content))

Online marketing serves as a cost-effective cornerstone for businesses by providing broad consumer reach through tools like SEO, social media, and content marketing. Additionally, data analytics enables companies to analyze customer behavior and refine their strategies for greater effectiveness.

In [20]:
prompt = f"""
<Instructions>
You are a marketing expert. Analyze the article within <Article> and provide recommendations based on the topics outlined in <Response_Format>.
</Instructions>

<Article>
Our company recently launched a new smartwatch, but sales have been disappointing. Most customers say the features aren't unique compared to competitors, and the price is too high for the value they receive.
</Article>

<Response_Format>
### Problem Analysis:
- [Summary of main issues]

### Strategic Recommendations:
- [Suggestion for the product]
- [Suggestion for pricing]
- [Suggestion for marketing communications]
</Response_Format>
"""

ai_msg = llm.invoke(prompt)

In [21]:
display(Markdown(ai_msg.content))

### Problem Analysis:
- The primary challenges are a lack of perceived product differentiation and a misalignment between the price point and the value proposition. Customers feel the smartwatch offers no unique advantages over competitors, leading to poor sales performance due to low perceived value relative to the cost.

### Strategic Recommendations:
- **Suggestion for the product**: Conduct a rapid competitive audit to identify underserved niche features (e.g., specific health metrics, battery life, or exclusive software integrations). If hardware changes are not immediately feasible, focus on software updates that introduce exclusive functionalities or partnerships to create a unique ecosystem that competitors do not offer.
- **Suggestion for pricing**: Implement a dynamic pricing strategy that includes introductory discounts, bundle deals (e.g., pairing the watch with a subscription service), or tiered pricing models to make the entry-level option more accessible. Alternatively, consider a "value-based" price adjustment to align closer with competitor averages until unique value propositions are clearly established.
- **Suggestion for marketing communications**: Shift the messaging focus from generic features to specific use-case benefits and emotional storytelling. Highlight any subtle differentiators through targeted content marketing, user testimonials, and comparison campaigns that directly address why this watch is worth the premium. Emphasize lifestyle integration and superior customer support as intangible value adders.

Explanation:

- `<Instructions>`: Sets the model's persona and primary objective.

- `<Article>`: Contains the raw data to be analyzed.

- `<Response_Format>`: Clearly outlines the desired structure of the output. This forces the model to organize its response systematically and address all specified points.



## Structured Output

`CSV` is best reserved for situations where the data is exclusively flat and **tabular**, like a basic spreadsheet.

`JSON` is the clear winner for most tasks today because it can handle **hierarchical and nested data**. This is essential for working with APIs, configurations, and any data that isn't a simple table. It also natively supports data types like integers, strings, and booleans, which simplifies processing.

### Output : CSV

In [31]:
# Example: Structured output (CSV)
prompt = """You are a helpful assistant.
**Task:** Convert the following customer list into a CSV string.
**Output Format:** The first row should contain the headers "Name" and "City".
The subsequent rows should contain the customer data, with values separated by commas.
Whole answer should be under the backtrick ```csv ... ```.
Response the final answer only.
**Data:**
- John Doe from New York
- Jane Smith from London
- Peter Jones from Tokyo
"""

ai_msg_csv = llm.invoke(prompt)
print(ai_msg_csv.content)

```csv
Name,City
John Doe,New York
Jane Smith,London
Peter Jones,Tokyo
```


#### Parsing CSV Output into a DataFrame

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [32]:
import re
import pandas as pd
import io

def csv_string_to_df(text: str) -> pd.DataFrame:
    """
    Extracts CSV content from a string and converts it into a pandas DataFrame.

    Args:
        text (str): The input string containing CSV content enclosed in ```csv...```.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the extracted data.
    """
    # Use a regex pattern to find the content between the delimiters
    match = re.search(r'```csv\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract the content from the first capturing group
        csv_content = match.group(1).strip()

        # Use io.StringIO to treat the string as a file
        data = io.StringIO(csv_content)

        # Read the "file" into a pandas DataFrame
        df = pd.read_csv(data)

        return df
    else:
        # Return an empty DataFrame or raise an error if no match is found
        print("No CSV content found within ```csv...``` delimiters.")
        return pd.DataFrame()

In [33]:
pd_object = csv_string_to_df(ai_msg_csv.content)
pd_object

,Name,City
0,John Doe,New York
1,Jane Smith,London
2,Peter Jones,Tokyo


### Output : JSON

In [34]:
# Example: Structured output (JSON)
prompt = """
You are a helpful assistant.
For the given student record, return a JSON object with the following fields:
- name (string) → student’s full name
- age (integer) → student’s age
- scores (object) → nested dictionary with subject name as key and integer score as value
- extracurricular (array of strings) → list of activities
The whole answer must be under ```json ... ```.
Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
Response the final answer only.
"""

ai_msg_json = llm.invoke(prompt)
print("Structured Output:\n", ai_msg_json.content)

Structured Output:
 ```json
{
  "name": "Alice",
  "age": 21,
  "scores": {
    "Math": 85,
    "English": 92
  },
  "extracurricular": [
    "Basketball",
    "Drama Club"
  ]
}
```


#### Parsing JSON Output into Dict

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [35]:
import re
import json

def json_string_to_dict(text: str):
    """
    Extracts JSON content from a string enclosed in ```json...```
    and parses it into a Python dict or list.

    Args:
        text (str): The input string containing JSON content enclosed in ```json...```.

    Returns:
        dict or list: Parsed JSON object (Python dict or list).
    """
    # Use regex to find JSON block
    match = re.search(r'```json\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract JSON content
        json_content = match.group(1).strip()

        try:
            return json.loads(json_content)
        except json.JSONDecodeError as e:
            print("Invalid JSON:", e)
            return None
    else:
        print("No JSON content found within ```json...``` delimiters.")
        return None

In [36]:
dict_output = json_string_to_dict(ai_msg_json.content)
dict_output

{'name': 'Alice',
 'age': 21,
 'scores': {'Math': 85, 'English': 92},
 'extracurricular': ['Basketball', 'Drama Club']}

In [37]:
dict_output['scores']['Math']

85

### Output : Pydantic Schema

- LangChain supports structured outputs, **allowing us to bind a schema (dict / JSON Schema / Pydantic) to the model**
  - and enforce responses to follow the defined schema (data type) instead of relying only on prompt wording.
- ***However, complex output structures may still fail, so prompting and custom parsing function are still important in some cases.***

Read more: [LangChain Docs – Structured Outputs](https://python.langchain.com/docs/concepts/structured_outputs/)


In [38]:
# pydantic schema

# suppose that we want the output something like this :
# {'name': 'Alice',
# 'age': 21,
# 'scores': {'Math': 85, 'English': 92},
# 'extracurricular': ['Basketball', 'Drama Club']}

# we can defined class (data fields) like this

from typing import Dict, List
from pydantic import BaseModel, Field

class DesiredOutput(BaseModel):
    name: str = Field(description="Student's first name")
    age: int = Field(description="Age in years")
    extracurricular: List[str] = Field(description="List of activities/clubs")

    #subject_scores: Dict[str, int] = Field(description="Key = subject, Value = scores (as a JSON object)") # This line cause an error. / Complex Data Structure (uncomment if you want to test it)

In [39]:
# Wrap LLM so it returns a DesiredOutput object directly
structured_llm = llm.with_structured_output(DesiredOutput)

In [40]:
prompt = """
You are a helpful assistant.
For the given student record, extract informations

Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
"""


# Generate output
results = structured_llm.invoke(prompt)
results

DesiredOutput(name='Alice', age=21, extracurricular=['Basketball', 'Drama Club'])

In [41]:
results.model_dump_json()

'{"name":"Alice","age":21,"extracurricular":["Basketball","Drama Club"]}'

## Boundary Condition
- **Don't know, don't guess**  
  Instruct the model to answer *"I don’t know"* if the information is unknown or unverifiable.  
  → Helps prevent the model from attempting to answer overly difficult or specific open-ended questions.  
  > Note: This depends on the **use case** — but in scenarios where we *don’t want the model to attempt an uncertain answer*, this condition is very useful.

- **Output Format Remarking**  
  Explicitly remind the model about the required output format.  
  → e.g., *"Don’t give any additional explanation, just output [format] only."*

In [42]:
# Example 1: Without boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?
"""

ai_msg = llm.invoke(prompt)
print("Without boundary condition:\n")
display(Markdown(ai_msg.content))

Without boundary condition:



Based on available public records and official archives up to mid-2024, **there is no official announcement titled “Announcement No. 2/2025” from the Meteorological Department regarding ‘Measures to Cope with the Early Arrival of Summer Storms’.**

### Reasoning:

1. **Temporal Impossibility**:  
   As of the current date (2024), the year **2025 has not yet occurred**. Therefore, no official government department—including any national Meteorological Department (e.g., India Meteorological Department, Hong Kong Observatory, etc.)—could have issued an announcement dated in 2025.

2. **Announcement Numbering Convention**:  
   Government announcements are typically numbered sequentially within a given year. “Announcement No. 2/2025” would imply it is the second official notice issued in the year 2025, which cannot exist before the year begins.

3. **Possible Confusion**:  
   You may be referring to:
   - A **past announcement** (e.g., Announcement No. 2/2024 or an earlier year).
   - A **hypothetical or fictional scenario**.
   - A **misremembered title or number** (e.g., “Announcement No. 2/2023” on pre-monsoon storms).
   - A **draft, proposal, or simulated exercise** not officially released.

### Recommendation:
- If you are referring to a **real, recent meteorological advisory**, please check the latest updates from your country’s official meteorological agency (e.g., India Meteorological Department, National Weather Service, etc.).
- If this is part of a **hypothetical, academic, or fictional context**, please clarify so I can assist accordingly.
- If you meant **Announcement No. 2/2024** (or another past year), please specify the country/region, and I can provide verified details if available.

Let me know how you’d like to proceed!

**Key Takeaways**:
- Without clear boundary conditions, an LLM will still attempt to generate an **answer—sometimes hallucinating content** **(especially in smaller models), and other times making an educated guess while acknowledging its uncertainty.**
- **Define clear boundary conditions and fallback responses so that uncertain cases can be reliably detected and handled in an automated pipeline.**
- This keeps your system consistent and predictable.

In [43]:
# Example 2: With boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?

If the answer is not known or cannot be verified, just reply: `None`.
"""

ai_msg = llm.invoke(prompt)
print("With boundary condition:\n", ai_msg.content)


With boundary condition:
 None


## Prompt Template

Prompt templates offer several benefits:

- **Consistency**: Ensure a consistent structure for your prompts across multiple interactions
- **Efficiency**: Easily swap out variable content without rewriting the entire prompt
- **Testability**: Quickly test different inputs and edge cases by changing only the variable portion
- **Scalability***: Simplify prompt management as your application grows in complexity
- **Version control**: Easily track changes to your prompt structure over time by keeping tabs only on the core part of your prompt, separate from dynamic inputs

### Example: Prompt Template in a Loop (Task: Sentiment Analysis)

Example Task: **Sentiment Analysis**

We used a prompt template with the approach **“run in a loop + change only variables”**.  
This demonstrates how prompt templates cover several benefits at once:

- **Consistency**: Every iteration uses the same prompt structure.  
- **Efficiency**: Only the variable `{text}` changes in each loop.  
- **Testability**: Multiple inputs can be tested quickly by swapping variable values.  
- **Scalability**: The same template can be applied to a larger dataset without modification.  
- **Version Control**: Easily track prompt versions against results.




In [44]:
!wget https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt

--2026-08-23 08:34:15--  https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt [following]
--2026-08-23 08:34:15--  https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 122071 (119K) [text/plain]
Saving to: ‘dev.txt’

dev.txt             100%[===================>] 119.21K  --.-KB/s    in 0.02s   

2026-08-23 08:34:15 (7.56 MB/

In [45]:
def read_xy_data(filename: str) -> tuple[list[str], list[int]]:
    x_data = []
    y_data = []
    with open(filename, 'r') as f:
        for line in f:
            label, text = line.strip().split(' ||| ')
            x_data.append(text)
            y_data.append(int(label))
    return x_data, y_data

In [47]:
x_test, y_test = read_xy_data('dev.txt')
x_test, y_test = x_test[:3], y_test[:3] # small size, respect the rate limit

For sentiment analysis, we will be using the following prompt:

```
Analyse the sentiment of the following text: ```text```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
```
LLMs nowaday usually have chain-of-thought baked in so they usually will output their reasoning before answering.

- It is important to tell the model not to output their explanation by including `**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
`
- Otherwise, it will not be easy to programmatically use the outputs.
Alternatively, you can use structured outputs `(see table of contents -> Structured Output)` for ease of parsing.

In [48]:
prompt_template = """
Analyse the sentiment of the following text: ```{x_input}```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**"""

In [50]:
import time
from tqdm.notebook import tqdm

output = []

# Practical use: add try/except for automatic retries,
# exponential backoff to handle temporary API/rate-limit errors,
# and sleep between requests to respect the provider's rate limits.

max_retries = 5
for sent in tqdm(x_test):
    prompt_filled = prompt_template.format(x_input=sent)
    print("prompt:", prompt_filled)  # debugging
    for attempt in range(max_retries):
        try:
            output_res = llm.invoke(prompt_filled).content.strip()
            print("response:", output_res)
            print("--" * 20)
            output.append(int(output_res))
            time.sleep(3)
            break

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt # exponential backoff
                print(
                    f"Request failed: {e}\n"
                    f"Retrying in {wait_time} seconds..."
                )
                time.sleep(wait_time)
            else:
                print(f"Failed after {max_retries} attempts.")
                output.append(0)

  0%|          | 0/3 [00:00<?, ?it/s]

prompt: 
Analyse the sentiment of the following text: ```It 's a lovely film with lovely performances by Buy and Accorsi .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```No one goes unindicted here , which is probably for the best .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: -1
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```And if you 're not nearly moved to tears by a couple of scenes , you 've got ice water in your veins .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------


In [51]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, output)

0.6666666666666666

## Additional: Temperature Setting

Temperature is a parameter that controls the randomness and diversity of an LLM’s output.
  - Keep it low if you are looking for more consistent and deterministic responses across repeated runs
  - Keep it high if you are looking for more diverse or creative responses.

### Approach 1 : Gemini

Temperature Range for Gemini-2.5-flash : 0-2 (default 1)

>Ref: https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/2-5-flash

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm_low_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [ ]:
# llm_high_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=2,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [ ]:
# # llm_low_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_low_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

In [ ]:
# # llm_high_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_high_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

### Approach 2 : Groq

In [55]:
llm_low_temp = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    temperature=0,
    reasoning_effort="none",
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [56]:
llm_high_temp = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    temperature=0.9,
    reasoning_effort="none",
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

**Low-temperature LLM :**

In [57]:
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_low_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(3.5)

Round 1 | response: Banking at your fingertips.
------------------------------------------------------------
Round 2 | response: Banking at your fingertips.
------------------------------------------------------------
Round 3 | response: Banking at your fingertips.
------------------------------------------------------------


**High-temperature LLM: **

In [58]:
# llm_high_temp
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_high_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(2) # Adding a 2-second delay to avoid rate limit error

Round 1 | response: Banking at the speed of life.
------------------------------------------------------------
Round 2 | response: Your wallet, anytime, anywhere.
------------------------------------------------------------
Round 3 | response: Banking at the speed of life.
------------------------------------------------------------


Summary

- Low temp → Reliable, consistent outputs. Useful for classification, extraction, or when you want reproducibility.
- High temp → Diverse, creative slogans. Useful for brainstorming, ideation, or when multiple fresh options are desired.

## Additional: Reasoning Effort Setting

Nowaday models support `thinking` mode:
- reasoning_effort="none" → faster, lower token usage, suitable for simple tasks
- reasoning_effort="default" → enables reasoning, useful for tasks that require multi-step thinking

In [61]:
prompt = """
A startup has two options for launching a new AI feature:

Option A:
- Faster to build
- Lower development cost
- Uses a less accurate model
- Can launch in 2 weeks

Option B:
- Higher development cost
- More accurate and reliable
- Requires 6 weeks to launch
- Better suited for long-term scaling

The company has limited budget but wants to build user trust.
Which option would you recommend, and why?

Answer in no more than 120 words.
"""

**Fast (No Reasoning)**

In [62]:
llm_fast = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none"
)

# No reasoning
start = time.perf_counter()
res_fast = llm_fast.invoke(prompt)
latency_fast = time.perf_counter() - start

display(Markdown("### No Reasoning"))
display(Markdown(res_fast.content))
print(f"Latency: {latency_fast:.2f} seconds")

### No Reasoning

I recommend Option A, provided the lower accuracy doesn’t compromise core functionality or safety. With a limited budget, survival is paramount. Launching in two weeks allows the startup to validate demand, generate early revenue, and gather real-world user data, which is crucial for securing further funding.

While Option B offers better long-term scaling, the higher cost and six-week delay pose significant financial risks for a cash-strapped startup. Trust is built through consistent delivery and responsiveness, not just initial perfection. By launching quickly with Option A, the company can demonstrate agility and iteratively improve the model based on actual user feedback. If the feature fails to gain traction, the company saves the substantial costs associated with Option B. Prioritize speed-to-market to ensure liquidity and validate the product-market fit before investing in premium accuracy.

Latency: 0.54 seconds


**Longer (Reasoning)**

In [63]:
llm_reasoning = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="default"
)

# No reasoning
start = time.perf_counter()
res_reasoning = llm_reasoning.invoke(prompt)
latency_reasoning = time.perf_counter() - start

display(Markdown("### No Reasoning"))
display(Markdown(res_reasoning.content))
print(f"Latency: {latency_reasoning:.2f} seconds")

### No Reasoning


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Startup context:** Launching a new AI feature
   - **Option A:** Faster (2 weeks), lower cost, less accurate model
   - **Option B:** Higher cost, more accurate/reliable, 6 weeks, better for long-term scaling
   - **Constraints:** Limited budget, wants to build user trust
   - **Task:** Recommend one option and explain why
   - **Constraint:** Max 120 words

2.  **Identify Key Factors:**
   - Limited budget favors Option A initially
   - User trust strongly favors Option B (accuracy/reliability)
   - AI features particularly need reliability to avoid user frustration/backlash
   - Long-term scaling favors Option B
   - Need to balance budget vs. trust/reliability

3.  **Weigh Options:**
   - Option A: Quick win, cheap, but poor accuracy could damage trust quickly, especially for AI. Might need costly rework later.
   - Option B: Higher upfront cost/time, but accuracy builds trust, supports scaling, and avoids reputational damage. Can be phased or funded strategically.
   - Given "wants to build user trust" is explicitly stated, Option B aligns better with that goal, despite budget constraints.

4.  **Formulate Recommendation (Mental Draft):**
   I recommend Option B. While Option A saves time and money upfront, AI features heavily depend on accuracy to build and maintain user trust. A less reliable model risks poor user experiences, negative feedback, and costly rework later. Option B’s higher accuracy and reliability directly support trust-building and long-term scaling. To manage the limited budget, the startup could phase development, seek seed funding, or launch a limited pilot to validate demand before full rollout. The short-term investment in quality prevents reputational damage and positions the product for sustainable growth, making it the smarter strategic choice.

5.  **Check Word Count:**
   Let's count words:
   I (1) recommend (2) Option (3) B. (4) While (5) Option (6) A (7) saves (8) time (9) and (10) money (11) upfront, (12) AI (13) features (14) heavily (15) depend (16) on (17) accuracy (18) to (19) build (20) and (21) maintain (22) user (23) trust. (24) A (25) less (26) reliable (27) model (28) risks (29) poor (30) user (31) experiences, (32) negative (33) feedback, (34) and (35) costly (36) rework (37) later. (38) Option (39) B’s (40) higher (41) accuracy (42) and (43) reliability (44) directly (45) support (46) trust-building (47) and (48) long-term (49) scaling. (50) To (51) manage (52) the (53) limited (54) budget, (55) the (56) startup (57) could (58) phase (59) development, (60) seek (61) seed (62) funding, (63) or (64) launch (65) a (66) limited (67) pilot (68) to (69) validate (70) demand (71) before (72) full (73) rollout. (74) The (75) short-term (76) investment (77) in (78) quality (79) prevents (80) reputational (81) damage (82) and (83) positions (84) the (85) product (86) for (87) sustainable (88) growth, (89) making (90) it (91) the (92) smarter (93) strategic (94) choice. (95)
   Total: ~95 words. Well under 120. Good.

6.  **Refine & Verify against Constraints:**
   - Recommends one option? Yes (B)
   - Explains why? Yes (trust, accuracy, long-term scaling, budget mitigation strategies)
   - Addresses limited budget + trust goal? Yes
   - Word count ≤ 120? Yes (~95)
   - Clear, professional tone? Yes

   All constraints met. Ready. Output matches the refined version.✅
</think>

I recommend **Option B**. While Option A is cheaper and faster, AI features require high accuracy to build and sustain user trust. A flawed model risks poor experiences, negative feedback, and expensive rework that ultimately costs more and damages reputation. Option B’s reliability directly supports trust-building and long-term scaling. To address budget constraints, the startup can phase development, secure a small bridge round, or run a targeted pilot to validate demand before full launch. Investing upfront in quality prevents reputational damage, reduces future technical debt, and positions the product for sustainable growth, making it the strategically sound choice despite the higher initial cost and timeline.

Latency: 2.51 seconds
